### Deep Learning (nnU-Net v2); Notebook 06

#### Objectif
Segmenter le thrombus voxel par voxel avec **nnU-Net v2** (package officiel, Isensee et al., *Nature Methods* 2021),
puis l'évaluer avec **la même FROC patient que O4** (sensibilité à 1/2/4/8 FP/patient sur les 33 porteurs)
pour comparer honnêtement classique vs deep learning.

#### Pourquoi nnU-Net officiel ?
- **Référence publiée** en segmentation médicale : il configure seul patch, normalisation CT, architecture et augmentations
  à partir des données et de la VRAM.
- **Mêmes folds que O4** (`splits_final.json`) → comparaison sur exactement les mêmes patients de test.
- **Évaluation honnête** : le modèle validé est le checkpoint final, pas le « meilleur » choisi sur le fold de test.
- Remplace `nnunet_pe.py` (implémentation maison : ~15 Go de RAM, checkpoint choisi sur le fold de test, pas de FROC).

#### Étapes
1. Environnement nnU-Net + GPU
2. Conversion FUMPE → format nnU-Net (+ contrôle d'alignement, folds O4)
3. Contrôle visuel d'un cas converti
4. Planification + prétraitement nnU-Net
5. Entraînement (test 5 époques, puis entraînement court)
6. Évaluation FROC + Dice + volume *(à venir)*

### 1. Environnement nnU-Net + GPU

Le kernel ne lit pas `scripts/nnunet_env.ps1` : on définit ici **les mêmes variables** (dossiers sous `data/nnunet/`, gitignoré).
`nnUNet_n_proc_DA = 3` limite les processus d'augmentation (défaut 12, trop pour 15 Go de RAM).
La fonction `nnunet(...)` lance les commandes officielles (`nnUNetv2_*.exe` de `NN_env`) et affiche leur sortie en direct.

> **Sortie attendue :** `CUDA : True | NVIDIA GeForce RTX 4060 Laptop GPU`.

In [ ]:
# Cellule 1 - Environnement nnU-Net + GPU
import os, sys, json, subprocess
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

# Mêmes valeurs que scripts/nnunet_env.ps1
NNUNET = ROOT / 'data' / 'nnunet'
os.environ['nnUNet_raw']          = str(NNUNET / 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = str(NNUNET / 'nnUNet_preprocessed')
os.environ['nnUNet_results']      = str(NNUNET / 'nnUNet_results')
os.environ['nnUNet_n_proc_DA']    = '3'      # garde-fou RAM
os.environ['PYTHONIOENCODING']    = 'utf-8'  # sorties des sous-processus lisibles
for k in ('nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results'):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

DATASET_ID = 501
DATASET    = 'Dataset501_FUMPE'
RAW_DS = Path(os.environ['nnUNet_raw']) / DATASET
PRE_DS = Path(os.environ['nnUNet_preprocessed']) / DATASET
RES_DS = Path(os.environ['nnUNet_results']) / DATASET
SCRIPTS = Path(sys.executable).parent        # NN_env\Scripts

def nnunet(cmd, *args):
    # Lance une commande nnU-Net officielle et affiche sa sortie en direct.
    full = [str(SCRIPTS / f'{cmd}.exe'), *map(str, args)]
    print('>', cmd, *full[1:], flush=True)
    p = subprocess.Popen(full, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding='utf-8', errors='replace', env=os.environ.copy())
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'{cmd} a échoué (code {p.returncode})')

import torch
print(f'Python : {sys.executable}')
print(f'torch {torch.__version__} | CUDA : {torch.cuda.is_available()} | '
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN GPU"}')
assert torch.cuda.is_available(), 'Pas de GPU : vérifier que le kernel est « NN_env (nnU-Net) »'

### 2. Conversion FUMPE → format nnU-Net

Logique dans `src/O5_deep_learning/nnunet_convert.py`. Pour chaque patient :
- **image** = HU bruts relus du DICOM (loader O1), NIfTI int16 avec le spacing réel — pas de fenêtrage, nnU-Net normalise lui-même ;
- **label** = `data/processed/PATxxx/mask.npy`, le GT corrigé utilisé par O3/O4 ;
- **contrôle d'alignement** : le DICOM fenêtré [-600, 400] doit redonner `volume.npy` (écart < 1e-3), sinon le patient est **refusé**.

Écrit aussi `dataset.json` (canal « CT ») et `splits_final.json` (**folds O4**). Un patient à la fois (~0,6 Go de RAM).

D'abord **PAT001 seul** (cellule 2a). Si `OK` avec un écart ~1e-7 → les 35 patients (cellule 2b, ~10 min).

In [ ]:
# Cellule 2a - Test de conversion sur PAT001
from src.O5_deep_learning.nnunet_convert import main as convert
convert(['PAT001'])

In [ ]:
# Cellule 2b - Conversion des 35 patients (les cas déjà convertis sont sautés)
convert([])

In [ ]:
# Cellule 2c - Rapport de conversion + folds
import pandas as pd
rep = pd.read_csv(RAW_DS / 'conversion_report.csv')
display(rep[['PatientID', 'status', 'max_diff', 'gt_voxels']] if 'max_diff' in rep else rep)

splits = json.loads((PRE_DS / 'splits_final.json').read_text())
for k, s in enumerate(splits):
    print(f'Fold {k} : {len(s["train"])} train | {len(s["val"])} val  {s["val"]}')
# Rappel : le fold 0 (O4) n'a que 3 patients de validation → tester plutôt sur le fold 1.

### 3. Contrôle visuel d'un cas converti

On relit les NIfTI écrits (pas les `.npy`) : la coupe axiale qui contient le plus de GT, fenêtre [-600, 400], GT en rouge.

> **Sortie attendue :** le contour rouge tombe **dans une artère pulmonaire** (gris clair, contrasté), pas dans l'air ni le poumon.

In [ ]:
# Cellule 3 - Contrôle visuel (NIfTI image + label)
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

PID = 'PAT001'
img = sitk.ReadImage(str(RAW_DS / 'imagesTr' / f'{PID}_0000.nii.gz'))
lbl = sitk.ReadImage(str(RAW_DS / 'labelsTr' / f'{PID}.nii.gz'))
vol, gt = sitk.GetArrayFromImage(img), sitk.GetArrayFromImage(lbl)   # (Z, Y, X)
print(f'{PID} : shape {vol.shape} | spacing (x,y,z) {img.GetSpacing()} | HU [{vol.min()}, {vol.max()}] | GT {int(gt.sum())} voxels')
print(f'HU médian sous le GT : {np.median(vol[gt > 0]):.0f}')

z = int(np.argmax(gt.sum(axis=(1, 2))))
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(vol[z], cmap='gray', vmin=-600, vmax=400); ax[0].set_title(f'{PID} coupe z={z}')
ax[1].imshow(vol[z], cmap='gray', vmin=-600, vmax=400)
ax[1].contour(gt[z], levels=[0.5], colors='red', linewidths=1); ax[1].set_title('GT (rouge)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
del vol, gt

### 4. Planification + prétraitement nnU-Net

`nnUNetv2_plan_and_preprocess` :
1. vérifie le dataset (`--verify_dataset_integrity` : géométrie image/label, labels 0/1) ;
2. extrait l'« empreinte » (spacings, intensités CT sous le GT) ;
3. choisit patch, batch et architecture pour **8 Go de VRAM** (cible par défaut du planner, qu'on ne touche pas) ;
4. prétraite les volumes.

On se limite à `3d_fullres` (la config 3D pleine résolution) et à **2 processus** (RAM). Durée : ~15–40 min, quelques Go sur F:.
`splits_final.json` (écrit à l'étape 2) est conservé.

In [ ]:
# Cellule 4a - Planification + prétraitement (3d_fullres uniquement, 2 processus)
nnunet('nnUNetv2_plan_and_preprocess', '-d', DATASET_ID, '-c', '3d_fullres',
       '-npfp', 2, '-np', 2, '--verify_dataset_integrity')

In [ ]:
# Cellule 4b - Ce que nnU-Net a choisi
plans = json.loads((PRE_DS / 'nnUNetPlans.json').read_text())
cfg = plans['configurations']['3d_fullres']
print(f"Patch (Z,Y,X)      : {cfg['patch_size']}")
print(f"Batch              : {cfg['batch_size']}")
print(f"Spacing cible (mm) : {cfg['spacing']}")
print(f"Normalisation      : {cfg['normalization_schemes']}")
print(f"splits_final.json toujours présent : {(PRE_DS / 'splits_final.json').exists()}")

### 5. Entraînement

**5a — Test de bout en bout, 5 époques, fold 1** (~10–20 min) : vérifie que tout tourne (GPU, RAM, validation, `--npz`)
et donne la **durée d'une époque** → on en déduit le budget réaliste avant la finale.
`--npz` garde les probabilités de validation : indispensables pour la FROC (étape 6).

**5b — Entraînement court (250 époques, fold 1)** : plusieurs heures. Mieux vaut le lancer **dans un terminal**
(le notebook n'a pas besoin de rester ouvert) :

```powershell
.\NN_env\Scripts\Activate.ps1
. .\scripts\nnunet_env.ps1
nnUNetv2_train 501 3d_fullres 1 -tr nnUNetTrainer_250epochs --npz
```
Reprise après interruption : même commande + `--c`. Les 4 autres folds ensuite, si le fold 1 est convaincant.

In [ ]:
# Cellule 5a - Test 5 époques sur le fold 1
nnunet('nnUNetv2_train', DATASET_ID, '3d_fullres', 1, '-tr', 'nnUNetTrainer_5epochs', '--npz')

In [ ]:
# Cellule 5b - Courbes d'entraînement (relancer pendant/après l'entraînement)
from IPython.display import Image, display
TRAINER = 'nnUNetTrainer_5epochs'   # puis 'nnUNetTrainer_250epochs'
FOLD = 1
fold_dir = RES_DS / f'{TRAINER}__nnUNetPlans__3d_fullres' / f'fold_{FOLD}'
png = fold_dir / 'progress.png'
display(Image(filename=str(png))) if png.exists() else print(f'Pas encore de courbes : {png}')
summ = fold_dir / 'validation' / 'summary.json'
if summ.exists():
    s = json.loads(summ.read_text())
    print(f"Dice moyen (validation fold {FOLD}) : {s['foreground_mean']['Dice']:.4f}")

### 6. Évaluation FROC + Dice + volume *(à venir)*

Principe, pour être comparable à O4 :
1. probabilités de validation (`.npz`) de chaque fold → composantes connexes = **candidats** ;
2. score d'un candidat = probabilité max dans la composante ; VP = candidat qui touche le GT (même règle qu'O3) ;
3. `froc_table` d'O4 → sensibilité patient à 1/2/4/8 FP/patient, à mettre face aux **45 % @4 FP** de la baseline.